In [3]:
#import all required libraries
import cv2
import numpy as np
import hashlib
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad

In [4]:
# Step 1: Derive a 16-byte AES key using SHA-256
def derive_key(userkey):
    return hashlib.sha256(userkey.encode()).digest()[:16]

# Step 2: Encrypt the message
def encrypt_message(msg, userkey):
    print("\nEncrypting message...")
    key = derive_key(userkey)
    cipher = AES.new(key, AES.MODE_CBC)
    ct = cipher.encrypt(pad(msg.encode(), AES.block_size))
    encrypted = cipher.iv + ct
    print("✔ Message encrypted.")
    return encrypted

In [5]:
# Step 3: Decrypt the message
def decrypt_message(cipher_bytes, userkey):
    print("\nDecrypting message...")
    key = derive_key(userkey)
    iv = cipher_bytes[:16]
    ct = cipher_bytes[16:]
    cipher = AES.new(key, AES.MODE_CBC, iv)
    decrypted = unpad(cipher.decrypt(ct), AES.block_size).decode()
    print("✔ Message decrypted.")
    return decrypted

# Step 4: Generate a blank image
def generate_image(width=300, height=300, color=(255, 255, 255)):
    print("\nGenerating RGB image with color:", color)
    image = np.zeros((height, width, 3), dtype=np.uint8)
    image[:] = color  # Fill entire image with the RGB color
    print("✔ RGB Image generated.")
    return image

In [6]:
# Step 5: Embed encrypted data into image
def embed_message(image, data_bytes, key):
    print("\nEmbedding message into image...")
    n = m = z = kl = 0
    for i in range(len(data_bytes)):
        image[n, m, z] = data_bytes[i] ^ ord(key[kl])
        n += 1
        if n >= image.shape[0]:
            n = 0
            m += 1
        if m >= image.shape[1]:
            m = 0
            z += 1
        if z >= 3:
            print("✘ Error: Image not big enough to hold data.")
            return None
        kl = (kl + 1) % len(key)
    print("✔ Message embedded into image.")
    return image

In [7]:
# Step 6: Extract data from image
def extract_message(image, key, length):
    print("\nExtracting message from image...")
    result = bytearray()
    n = m = z = kl = 0
    for i in range(length):
        byte = image[n, m, z] ^ ord(key[kl])
        result.append(byte)
        n += 1
        if n >= image.shape[0]:
            n = 0
            m += 1
        if m >= image.shape[1]:
            m = 0
            z += 1
        if z >= 3:
            break
        kl = (kl + 1) % len(key)
    print("✔ Message extracted from image.")
    return result

In [8]:
# Step 7: Main function with user input
def main():
    print("\n====== SECURE AES + STEGANOGRAPHY SYSTEM ======\n")

    message = input("Enter the message to encrypt and hide: ")
    password = input("Enter a strong password: ")

    print(f"\n Original Message: {message}")
    print(f"Encryption Key: {password}")

    encrypted_data = encrypt_message(message, password)
    print(f"\n Encrypted Data (in bytes): {encrypted_data}")

    base_image = generate_image()

    stego_image = embed_message(base_image.copy(), encrypted_data, password)

    if stego_image is not None:
        cv2.imwrite("stego_image.png", stego_image)
        print("\n Image saved as 'stego_image.png' with hidden message.")

        extracted_data = extract_message(stego_image, password, len(encrypted_data))
        print(f"\n Extracted Encrypted Bytes: {extracted_data}")

        decrypted_message = decrypt_message(extracted_data, password)
        print(f"\n Decrypted Message: {decrypted_message}")

    print("\n Process Complete.")

if __name__ == "__main__":
    main()


====== SECURE AES + STEGANOGRAPHY SYSTEM ======


 Original Message: JAI HIND
Encryption Key: R!te$h

Encrypting message...
✔ Message encrypted.

 Encrypted Data (in bytes): b'5s\x10?R\xfa\x06\xa6\xa9\xdf\x01\xb1\xd7\x07\x01v\xb9Kp\xce\x1f:\x99Dd\xa1\xfc\x8a\xf4\x84\xc2\xf8'

Generating RGB image with color: (255, 255, 255)
✔ RGB Image generated.

Embedding message into image...
✔ Message embedded into image.

 Image saved as 'stego_image.png' with hidden message.

Extracting message from image...
✔ Message extracted from image.

 Extracted Encrypted Bytes: bytearray(b'5s\x10?R\xfa\x06\xa6\xa9\xdf\x01\xb1\xd7\x07\x01v\xb9Kp\xce\x1f:\x99Dd\xa1\xfc\x8a\xf4\x84\xc2\xf8')

Decrypting message...
✔ Message decrypted.

 Decrypted Message: JAI HIND

 Process Complete.
